In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()
HuggingFaceApi = os.getenv('HF_TOKEN')
groq_api_key = os.getenv("GROQ_API_KEY")

In [2]:
from langchain_groq import ChatGroq
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


loader = PyPDFLoader(r'pdf_directory\APJ_Speech.pdf')
docs = loader.load()

llm = ChatGroq(model = 'llama3-8b-8192', api_key=groq_api_key, temperature=1, max_tokens=1024)

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 5000, chunk_overlap = 200)
documents = text_splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

c:\vs_code\Genai\genai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain_community.vectorstores import FAISS
vector_databse = FAISS.from_documents(docs, embeddings)

In [5]:
retriever = vector_databse.as_retriever()
retriever.invoke("who is apj abdul kalam ")

[Document(metadata={'source': 'pdf_directory\\APJ_Speech.pdf', 'page': 0}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who are the future wealth of our country. During my intera ction at \nRashtrapati Bhavan in Delhi and at every state and union territor y as well as through my \nonline interactions, I have many unique experiences to share with you, which signi

In [6]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
    answer the questions based on the provided context only.
    Please provide the most accurate  response based on the question 
    <context>
    {context}
    <context>
    Question:{input}
    
    """
        )

In [7]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

document_chain = create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(retriever, document_chain)


In [8]:
user_prompt = 'calculate the age of apj abdul kalam'
response = retrieval_chain.invoke({'input': user_prompt})

In [9]:
response

{'input': 'calculate the age of apj abdul kalam',
 'context': [Document(metadata={'source': 'pdf_directory\\APJ_Speech.pdf', 'page': 0}, page_content='A P J Abdul Kalam Departing speech \n \n \nFriends, I am delighted to address you all, in the country and those livi ng abroad, after \nworking with you and completing five beautiful and eventful years in Rashtrapati \nBhavan. Today, it is indeed a thanks giving occasion. I would like to narr ate, how I \nenjoyed every minute of my tenure enriched by the wonderful assoc iation from each one \nof you, hailing from different walks of life, be it politics, sci ence and technology, \nacademics, arts, literature, business, judiciary, administration, local bodies, farming, \nhome makers, special children, media and above all from the youth and st udent \ncommunity who are the future wealth of our country. During my intera ction at \nRashtrapati Bhavan in Delhi and at every state and union territor y as well as through my \nonline interactions,

In [10]:
print(response['answer'])

There is no information provided in the context about the age of APJ Abdul Kalam.


In [11]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    retrieval_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

In [14]:
conversational_rag_chain.invoke(
    {"input": "what is the age of the apj abdul kalam"},
    config={
        "configurable": {"session_id": "abc123"}
    },  # constructs a key "abc123" in `store`.
)["answer"]

'There is no mention of the age of A.P.J. Abdul Kalam in the provided context.'

In [15]:
from langchain_core.messages import AIMessage, HumanMessage

for message in store["abc123"].messages:
    if isinstance(message, AIMessage):
        prefix = "AI"
    else:
        prefix = "User"

    print(f"{prefix}: {message.content}\n")

User: who is apj abdul kalam

AI: The answer is: Dr. A. P. J. Abdul Kalam

User: what is the age of the apj abdul kalam

AI: There is no mention of the age of A.P.J. Abdul Kalam in the provided context.

